In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Clone mã nguồn gốc của bài báo
!git clone https://github.com/Dingdong-Inc/frn-50k-baseline.git
%cd /content/frn-50k-baseline/demand_forecasting/SSA

Mounted at /content/drive
Cloning into 'frn-50k-baseline'...
remote: Enumerating objects: 334, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 334 (delta 12), reused 13 (delta 9), pack-reused 311 (from 1)
Receiving objects: 100% (334/334), 269.43 KiB | 5.39 MiB/s, done.
Resolving deltas: 100% (88/88), done.
/content/frn-50k-baseline/demand_forecasting/SSA


In [ ]:
!pip install -q datasets pyarrow pandas

import os
from datasets import load_dataset

# Tạo đúng cấu trúc thư mục như Task 6 quy định
os.makedirs("/content/drive/MyDrive/PTLDKD/data", exist_ok=True)

# Tải bộ dữ liệu chính thức từ Hugging Face
dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K")

# Xuất ra 2 file train.parquet và eval.parquet
dataset["train"].to_parquet("/content/drive/MyDrive/PTLDKD/data/train.parquet")
dataset["eval"].to_parquet("/content/drive/MyDrive/PTLDKD/data/eval.parquet")
print("Đã xuất xong train.parquet và eval.parquet!")

README.md:   0%|          | 0.00/4.88k [00:00<?, ?B/s]

data/train.parquet: reconstructing file:   0%|          |  0.00B /  106MB            

data/train.parquet: downloading bytes:           |  0.00B            

data/eval.parquet: reconstructing file:   0%|          |  0.00B / 8.44MB            

data/eval.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4500000 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/350000 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/24 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Đã xuất xong train.parquet và eval.parquet!


In [ ]:
import pandas as pd

# 1. Clone repo gốc của bài báo
!git clone https://github.com/Dingdong-Inc/frn-50k-baseline.git

# 2. Kiểm tra dữ liệu đã lưu trên Drive
train_path = "/content/drive/MyDrive/PTLDKD/data/train.parquet"
eval_path = "/content/drive/MyDrive/PTLDKD/data/eval.parquet"

df_train = pd.read_parquet(train_path)
df_eval = pd.read_parquet(eval_path)

print(f"Train: {len(df_train):,} dòng | {df_train[['store_id', 'product_id']].drop_duplicates().shape[0]:,} cặp")
print(f"Eval : {len(df_eval):,} dòng | {df_eval[['store_id', 'product_id']].drop_duplicates().shape[0]:,} cặp")

# Lọc điều kiện đánh giá của Task 8: stock_hour6_22_cnt == 0
valid_eval = df_eval[df_eval['stock_hour6_22_cnt'] == 0]
print(f"Số dòng eval đủ điều kiện đánh giá (stock_hour6_22_cnt == 0): {len(valid_eval):,}")

Cloning into 'frn-50k-baseline'...
remote: Enumerating objects: 334, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 334 (delta 12), reused 13 (delta 9), pack-reused 311 (from 1)
Receiving objects: 100% (334/334), 269.43 KiB | 7.48 MiB/s, done.
Resolving deltas: 100% (88/88), done.
Train: 4,500,000 dòng | 50,000 cặp
Eval : 350,000 dòng | 50,000 cặp
Số dòng eval đủ điều kiện đánh giá (stock_hour6_22_cnt == 0): 206,562


In [9]:
import pandas as pd
import numpy as np
import os

os.makedirs("/content/task08_outputs/full", exist_ok=True)

# 1. Chỉ đọc đúng các cột cần thiết để tiết kiệm RAM trên Colab
cols = ['store_id', 'product_id', 'dt', 'sale_amount', 'stock_hour6_22_cnt', 'discount', 'holiday_flag', 'precpt']
train_df = pd.read_parquet("/content/drive/MyDrive/PTLDKD/data/train.parquet", columns=cols)
eval_df = pd.read_parquet("/content/drive/MyDrive/PTLDKD/data/eval.parquet", columns=cols)

all_data = pd.concat([train_df, eval_df], axis=0, ignore_index=True)
del train_df, eval_df  # Giải phóng RAM ngay lập tức

all_data = all_data.sort_values(by=['store_id', 'product_id', 'dt']).reset_index(drop=True)

# 2. Tạo đặc trưng lịch và thời tiết giống hệt ssa_forecasting.py
dt_series = pd.to_datetime(all_data['dt'])
all_data['time_idx'] = (dt_series - dt_series.min()).dt.days
all_data['dow'] = dt_series.dt.dayofweek
all_data.loc[all_data['dt'].isin(['2024-05-02', '2024-05-03', '2024-05-04']), 'holiday_flag'] = 0
all_data.loc[all_data['holiday_flag'] == 1, 'dow'] = 5
all_data.loc[(all_data['holiday_flag'] == 0) & (all_data['dow'].isin([5, 6])), 'dow'] = 0
all_data['is_precpt'] = (all_data['precpt'] > 3.5).astype(int)

# 3. Tính trọng số SSA
x_cont = all_data[['time_idx', 'discount', 'sale_amount']].values.reshape(-1, 97, 3)
x_cat = all_data[['dow', 'holiday_flag', 'is_precpt']].values.reshape(-1, 97, 3)

max_encoder_length = 90
holiday_label = 5

date_distance = 18 - x_cont[:, :max_encoder_length, 0][:, None, :] / 5
discount_distance = 1 - np.abs(x_cont[:, max_encoder_length:, 1][..., None] - x_cont[:, :max_encoder_length, 1][:, None, :])

decoder_dow = x_cat[:, max_encoder_length:, 0][..., None]
encoder_dow = x_cat[:, :max_encoder_length, 0][:, None, :]
dow_distance = decoder_dow - encoder_dow

decoder_precpt = x_cat[:, max_encoder_length:, 2][..., None]
encoder_precpt = x_cat[:, :max_encoder_length, 2][:, None, :]
precpt_distance = decoder_precpt - encoder_precpt

date_distance = np.where((decoder_dow == holiday_label).astype(int) - (encoder_dow == holiday_label).astype(int) == 0, date_distance - 3, date_distance)
date_distance = np.where((decoder_dow != holiday_label) & (encoder_dow == holiday_label), date_distance + 3, date_distance)
date_distance = np.where(dow_distance == 0, date_distance - 3, date_distance)
date_distance = np.where((precpt_distance == 0) & (decoder_precpt == 1), date_distance - 3, date_distance)
date_distance = np.where(precpt_distance != 0, date_distance + 3, date_distance)
date_distance = np.clip(date_distance, a_min=1, a_max=18)

weight = (18 - date_distance) * (discount_distance ** 2)
weight = np.exp(weight) / np.exp(weight).sum(axis=-1, keepdims=True)

# 4. Chia nhóm High Sale / Low Sale dựa trên psd
mu = all_data[all_data['dt'] <= '2024-06-25'].groupby(['store_id', 'product_id'], as_index=False)['sale_amount'].mean()
mu.rename(columns={'sale_amount': 'psd'}, inplace=True)

pdf = all_data[all_data['dt'] >= '2024-06-26'].copy()
pdf = pdf.merge(mu, on=['store_id', 'product_id'], how='left')

target = x_cont[:, :max_encoder_length, 2:3]
index = encoder_dow.squeeze()[..., None] == np.arange(6)
mean_sale = np.nanmean(np.where(index, target, np.nan), axis=1) + 0.001
season_ratio = np.nanmedian(mean_sale[..., None] / mean_sale[:, None, ...], axis=0)
ratio = season_ratio[decoder_dow, encoder_dow]
pred = (weight * ratio) @ target
pdf['ssa_pred_sale_amount'] = pred.reshape(-1)

# 5. Đánh giá 3 nhóm theo đúng hàm evaluation của tác giả
def evaluate_group(df_eval_pred, condition):
    wape_list, mae_list, wpe_list = [], [], []
    sub = df_eval_pred.query('stock_hour6_22_cnt == 0').query(condition)
    for _, subdf in sub.groupby('dt'):
        abs_err = (subdf['sale_amount'] - subdf['ssa_pred_sale_amount']).abs()
        wape_list.append(abs_err.sum() / subdf['sale_amount'].sum())
        mae_list.append(abs_err.mean())
        wpe_list.append((subdf['ssa_pred_sale_amount'] - subdf['sale_amount']).sum() / subdf['sale_amount'].sum())
    return round(np.mean(wape_list), 4), round(np.mean(wpe_list), 4), round(np.mean(mae_list), 4)

paper_ref = {
    "Overall":   {"cond": "psd>=0", "wape": 31.97, "wpe": -10.50},
    "Low Sale":  {"cond": "psd<1",  "wape": 38.33, "wpe": -8.03},
    "High Sale": {"cond": "psd>=1", "wape": 27.40, "wpe": -12.26}
}

log_lines = ["group\tdemand\twape\twpe\tmae"]
csv_rows = []

for group_name, info in paper_ref.items():
    wape, wpe, mae = evaluate_group(pdf, info["cond"])
    log_lines.append(f"{info['cond']}\tssa_pred_sale_amount\t{wape}\t{wpe}\t{mae}")

    wape_pct = round(wape * 100, 2)
    wpe_pct = round(wpe * 100, 2)
    csv_rows.append({
        "config_id": "SSA_RawSales",
        "group": group_name,
        "wape_pct": wape_pct,
        "wpe_pct": wpe_pct,
        "mae": mae,
        "paper_wape_pct": info["wape"],
        "paper_wpe_pct": info["wpe"],
        "delta_wape_pp": round(wape_pct - info["wape"], 2),
        "delta_wpe_pp": round(wpe_pct - info["wpe"], 2),
        "status": "completed"
    })

# 6. Lưu file log và file CSV bàn giao
log_content = "\n".join(log_lines) + "\n"
with open("/content/task08_outputs/full/ssa_raw_sales.log", "w") as f:
    f.write(log_content)

df_metrics = pd.DataFrame(csv_rows)
df_metrics.to_csv("/content/task08_outputs/ssa_raw_sales_metrics.csv", index=False)

print("=== NỘI DUNG FILE LOG ===")
print(log_content)
print("=== BẢNG KẾT QUẢ CSV ===")
display(df_metrics)

=== NỘI DUNG FILE LOG ===
group	demand	wape	wpe	mae
psd>=0	ssa_pred_sale_amount	0.3197	-0.105	0.3982
psd<1	ssa_pred_sale_amount	0.3833	-0.0803	0.2691
psd>=1	ssa_pred_sale_amount	0.274	-0.1226	0.7761

=== BẢNG KẾT QUẢ CSV ===


,config_id,group,wape_pct,wpe_pct,mae,paper_wape_pct,paper_wpe_pct,delta_wape_pp,delta_wpe_pp,status
0,SSA_RawSales,Overall,31.97,-10.50,0.3982,31.97,-10.50,0.0,0.0,completed
1,SSA_RawSales,Low Sale,38.33,-8.03,0.2691,38.33,-8.03,0.0,0.0,completed
2,SSA_RawSales,High Sale,27.40,-12.26,0.7761,27.40,-12.26,0.0,0.0,completed
